# Data worth, for nearly free

_Which analyte is worth paying the lab for?_

Back in [`part1_03_obs_weights_and_truth`](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) we conditioned the history match on five species &mdash; SO₄, O₂, NO₃, pH and temperature (`so4`, `o0`, `no3`, `ph`, `tmp`) &mdash; and **deliberately held back the major cations** (Ca, Mg, Na, K, Fe). We made a promise there: we would come back and ask whether measuring those cations would actually have *helped*. This is where we keep it.

The question is not academic. Field chemistry costs money. Every analyte you add to the sampling schedule is more lab time, more boreholes sampled more often, more budget. A modeller is constantly asked, in one form or another: **is this measurement worth it?** — will spending on it actually sharpen the decision, or just add numbers to a spreadsheet that the forecast never feels.

"Data worth" is the formal name for that question, and the answer is decision-first: a measurement is worth something *only if it would have narrowed the thing we are deciding on*. Here that thing is the canonical forecast — peak SO₄ at the supply well (`wellopt`) over the supply period, whose distribution sets the treatment capacity the operator designs to its P95. A new measurement is worth paying for if, and only if, adding it to the history match would have tightened the posterior forecast distribution: less spread, a lower P95, less treatment capacity bought against uncertainty we could have resolved for the price of a lab analysis.

Where this notebook sits in the sequence:

- [`../part1_05_dsi_basics/dizon_dsi_basics.ipynb`](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) trained the DSI emulator on the prior Monte Carlo and conditioned it on the five conditioning species — that is the *baseline* history match we measure everything against here.
- [`../part1_06_full_model_check/`](../part1_06_full_model_check/) validated that DSI posterior against the full model.
- [`../part1_03_obs_weights_and_truth/`](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) is where the held-back cations were set aside and the synthetic truth was chosen.
- [`../part1_08_optimization/`](../part1_08_optimization/) uses the same emulator to value held-back *decisions*; this notebook values held-back *data*.

> **Skeleton notebook.** The thinned prior Monte Carlo ensemble and the conditioned DSI artifacts this workflow consumes are produced upstream (`part1_04`, `part1_05`) and by `prebaked/`; the per-subset comparison has not been baked yet. The code cells below are headed `TODO` and carry commented stub calls against the vendored `pyemu` DSI API. They are the intended *shape* of the workflow, not a runnable notebook. When the upstream notebooks and `prebaked/` ship, the stubs become live code.

> The one thing this notebook will **never** do is run the full model. Every number it produces comes from re-fitting and re-conditioning the DSI emulator on different slices of observations the prior ensemble *already produced*. That is the whole point — see [why emulation makes this cheap](#Why-emulation-makes-this-cheap), below.

### Admin

We lean on the vendored dependency trees (`flopy`, `pyemu`) that ship with this repository; the asserts below fail loudly if a different `pyemu` is on the path. `herebedragons` (imported as `hbd`) holds the shared plotting and bookkeeping helpers, and every workspace this notebook creates lives *inside this notebook's own directory*.

In [ ]:
import os
import sys
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import flopy
import pyemu
from pyemu.emulators import DSI

warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
import herebedragons as hbd

Confirm we are running the vendored dependencies and not something else `pip` dragged in:

In [ ]:
assert "dependencies" in flopy.__file__, flopy.__file__
assert "dependencies" in pyemu.__file__, pyemu.__file__

**Prerequisite check.** Everything here is built on the fitted DSI emulator and the prior Monte Carlo observation ensemble from upstream. We need:

1. the prior Monte Carlo run results — the tracked, thinned `prebaked/prior_mc_obs_ensemble.jcb` artifact, or a full run in the repo-root `master_priormc` — and
2. the runstore-prepared DSI template directory (`dsi_template/` with `dsi.pst`) from `part1_05`.

If either is missing, stop and run the upstream notebook first.

In [ ]:
# resolve the prior MC source in the canonical order used across the series:
#   1. the tracked, thinned prebaked obs ensemble (../../prebaked/prior_mc_obs_ensemble.jcb)
#   2. a full prior MC run into the repo-root master dir (../../master_priormc)
prebaked_oe = Path("..") / ".." / "prebaked" / "prior_mc_obs_ensemble.jcb"
priormc_d = Path("..") / ".." / "master_priormc"
if not (prebaked_oe.exists() or (priormc_d / "pest.pst").exists()):
    raise Exception(
        "you need to run the '../part1_04_prior_mc/dizon_prior_mc.ipynb' notebook first "
        "— it produces the prior Monte Carlo ensemble this notebook re-slices "
        f"(looked for the prebaked artifact '{prebaked_oe}' and the master dir '{priormc_d}')"
    )

dsi_t_d = Path("..") / "part1_05_dsi_basics" / "dsi_template"
if not (dsi_t_d / "dsi.pst").exists():
    raise Exception(
        "you need to run the '../part1_05_dsi_basics/dizon_dsi_basics.ipynb' notebook first "
        "— it builds the runstore-prepared dsi_template/ and the baseline conditioned emulator"
    )

Worker count for the parallel conditioning runs. Each observation subset triggers a PESTPP-IES conditioning *of the emulator* — fast, but there are several of them, so we parallelise. Physical cores only:

In [ ]:
num_workers = psutil.cpu_count(logical=False)
num_workers

## What data worth means here

The classic data-worth experiment compares two history matches that differ only in their observations, and reads the difference *in the forecast*. There are two directions:

- **Adding** observations and watching the forecast uncertainty shrink — "what would this extra data buy us?"
- **Removing** observations and watching it grow — "how much is the data we already collect actually doing?"

Both reduce to the same operation: re-condition on a different observation set and compare the posterior forecast spread. We will do mostly the *adding* kind — the held-back cations are the headline — but the removing kind (drop a site, drop a species) answers the budget question from the other side: if a measurement could be cut without the forecast noticing, you are paying for it for nothing.

The metric is the posterior **forecast** distribution — *not* how well each subset fits its own observations. A subset that fits beautifully but leaves the supply-well sulfate forecast just as wide as the prior has bought us nothing we can decide on. So for every subset we summarise the same single number: the spread of peak SO₄ at the supply well over the supply period (e.g. the 5th–95th percentile range, or the standard deviation), and where its P95 sits. A *narrower* posterior — and especially a lower P95, the number treatment capacity is designed to — is the worth.

## Why emulation makes this cheap

Done the conventional way, data worth is brutally expensive. To value one candidate observation set you would re-run the whole history match against the full model — a fresh PESTPP-IES ensemble, hundreds of ~6 min/run DIZON evaluations, *per subset*. With a handful of subsets that is days of compute to answer a question that is, in the end, advisory: it tells you what to go and measure, it does not change the model.

That cost is exactly why conventional data-worth analysis is so often skipped on reactive-transport models, and exactly the gap emulation closes.

Here is the trick. The prior Monte Carlo already ran the full model once for every realisation, and recorded **all** the outputs — every species at every site at every time, the held-back cations included. The DSI emulator learned the joint distribution of *all* of those outputs. So:

- Choosing a different observation subset to condition on is just choosing **different columns** of an ensemble that already exists. No new forward runs.
- Re-fitting DSI on those columns and re-conditioning with PESTPP-IES runs in **seconds to minutes on the emulator**, not days on the model.

The data we "held back" was never actually un-simulated — the prior MC computed the cations for free, as a side effect of running the model at all. Emulation lets us cash that in. **This is the dividend:** the expensive ensemble was paid for once, upstream, and data worth falls out of it for nearly nothing.

One honest caveat, stated up front so it does not get lost. We are valuing the cations *against the synthetic truth* — the cation "measurements" we condition on are this truth realisation's own cation values, perturbed by the noise model. So this experiment answers "would these measurements have helped *in this synthetic world*?", which is the right question for teaching the method and a fair guide for the real campaign — but it is not a promise about the field, where structural error muddies what any measurement can constrain. The [real-data capstone](../part1_09_real_data_capstone/) is where that distinction comes home.

## The baseline: the history match we already have

Everything is measured against the conditioned DSI posterior from `part1_05` — the one trained and conditioned on the five conditioning species. That is our reference forecast distribution. Load it, pull out the supply-well sulfate forecast, and summarise its spread. Every subset below gets compared back to this.

In [ ]:
# TODO: load the baseline conditioned DSI posterior and its forecast spread (PENDING upstream).
#
# The baseline is what part1_05 produced: DSI fit on prior MC (minus the synthetic truth),
# conditioned on the five conditioning species. We reuse its posterior obs ensemble.
#
# dsi_pst = pyemu.Pst(str(dsi_t_d / "dsi.pst"))
# post_iters = sorted(
#     int(f.split(".")[1]) for f in os.listdir(dsi_t_d)
#     if f.startswith("dsi.") and ".obs." in f and f.split(".")[1].isdigit()
# )
# oe_base = pyemu.ObservationEnsemble.from_binary(
#     pst=dsi_pst, filename=str(dsi_t_d / f"dsi.{max(post_iters)}.obs.jcb")
# )
# oe_base.shape

Define the forecast once and reuse it for every subset — peak SO₄ at the supply well over the supply period, the same definition `part1_05` used. The forecast spans all three supply-well screens (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`); the forecast species is `so4`; the supply period is 308–728 d. "Peak" is the maximum over all those screens and supply-period times, taken per realisation, so the forecast is a *distribution* across the ensemble.

In [ ]:
# TODO: assemble the forecast obs list and a peak-extraction helper (no model run).
#
# SUPPLY_START, SUPPLY_END = 308.0, 728.0   # supply period (days); forecast lives here
# SO4_MOLAR_MASS   = 96.06                  # g/mol, for the mol/L <-> mg/L conversion
#
# obs = dsi_pst.observation_data
# fore = obs[
#     (obs.obsid.isin(["welopt-ly1", "welopt-ly3", "welopt-ly5"]))  # all supply-well screens
#     & (obs.variable == "so4")
#     & (obs.time.astype(float) >= SUPPLY_START)
#     & (obs.time.astype(float) <= SUPPLY_END)
# ]
# forecast_obsnmes = fore.obsnme.tolist()
#
# def peak_so4_mgl(oe):
#     """Per-realisation peak supply-well SO4 (mg/L) over the supply period."""
#     peak_moll = oe.loc[:, forecast_obsnmes].max(axis=1)   # max over times+layers
#     return peak_moll * SO4_MOLAR_MASS / 1e-3              # mol/L -> mg/L

## The species question RTM modellers actually face

The headline experiment is the one the field campaign cares about. The pyrite-oxidation network couples the redox drive (O₂, NO₃ consumed) to the product (SO₄) and to a *cascade of secondary signals*: the cations released or exchanged as the front passes — calcium and sodium from carbonate dissolution and ion exchange, iron from the pyrite itself. Intuitively those cations carry information about the same reaction that drives the forecast. **Do they carry enough to be worth sampling?**

That is the analyte question stated concretely: for each held-back cation, condition the emulator on the five baseline species *plus that cation*, and see whether the supply-well sulfate forecast tightens. A cation that moves the forecast is one the lab budget should buy; one that does not is one we can keep holding back without regret.

Pin down the species sets, matching the definitions from [`part1_03`](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) so the baseline here is *exactly* the baseline we conditioned on there. Note the model only simulates a subset of the conceptual cation list — we filter to species the prior ensemble actually produced, so we never invent an observation.

In [ ]:
# conditioning species (the baseline history match; o0 is dissolved O2 in PHREEQC notation)
COND_SPECIES   = ['so4', 'o0', 'no3', 'ph', 'tmp']
# the major cations we held back in part1_03 (conceptual list)
HELD_BACK_CATS = ['ca', 'mg', 'na', 'k', 'fe', 'fe2']

# TODO: keep only the held-back species the model actually outputs (PENDING obs load).
#
# present = set(obs.variable.unique())
# cond_present = [s for s in COND_SPECIES if s in present]
# cats_present = [s for s in HELD_BACK_CATS if s in present]
# print("conditioning species present:", cond_present)   # expect: so4 o0 no3 ph tmp
# print("held-back cations present   :", cats_present)   # expect: ca na fe fe2 (mg, k not simulated)

## The recipe: one emulator re-fit per subset

Every data-worth case is the same three steps, differing only in *which observations carry weight*:

1. **Choose the subset.** A list of conditioning observations — the baseline five species, plus or minus a cation, a site, or a species, optionally over a shorter or longer monitoring window.
2. **Re-fit and re-condition DSI** on that subset. We re-fit on the *same* prior MC training ensemble (the emulator already knows all the outputs); what changes is which observations get non-zero weight and noise when PESTPP-IES conditions the emulator. Same transform, same energy threshold, same `ies_multimodal_alpha=0.99` as the baseline — only the observation set moves, so the comparison is clean.
3. **Read the forecast spread** with `peak_so4_mgl` and compare back to the baseline.

We wrap that in a single helper so each subset is one line.

First, a function that builds the conditioning weights and noise for a given (species, sites, time-window) selection — the same weighting logic `part1_03` used, just parameterised by the subset. Weights are zero outside the window and for any species not in the subset; species-specific noise (proportional with a floor for concentrations, absolute for pH and temperature) sets the standard deviations; per site:species phi factors keep any one group from dominating.

In [ ]:
# TODO: build per-subset conditioning weights + noise on the DSI control file (PENDING).
#
# def weight_subset(dpst, species, sites=None, t_start=0.0, t_end=252.0):
#     """Set weights/noise on the DSI pst for one conditioning subset.
#
#     species : list of variable codes to condition on (e.g. COND_SPECIES + ['ca'])
#     sites   : list of obsids to keep (default: all conditioning sites)
#     t_start, t_end : monitoring window (default: full history period, 0-252 d)
#     """
#     dobs = dpst.observation_data
#     dobs['time'] = dobs['time'].astype(float)
#     dobs['weight'] = 0.0
#     sel = (
#         dobs.variable.isin(species)
#         & (dobs.time >= t_start) & (dobs.time <= t_end)
#         & (dobs.obsval < 1e30)            # 1e30 is the no-data flag
#     )
#     if sites is not None:
#         sel &= dobs.obsid.isin(sites)
#     nz = dobs.loc[sel]
#     # species-specific noise: proportional+floor for concs, absolute for pH/Tmp
#     # (reuse the exact noise table from part1_03 so subsets are comparable)
#     # dobs.loc[nz.index, 'standard_deviation'] = ...   # per part1_03
#     # dobs.loc[nz.index, 'weight'] = 1.0 / dobs.loc[nz.index, 'standard_deviation']
#     return dpst

Then the worth helper itself: copy the runstore DSI template, apply the subset weighting, set the truth values + noise ensemble, condition with PESTPP-IES on the emulator, and return the posterior forecast spread.

In [ ]:
# TODO: condition the emulator on one subset and return its forecast spread (PENDING).
#
# def worth_of(name, species, sites=None, t_start=0.0, t_end=252.0, truth=None):
#     """Re-condition DSI on one obs subset; return posterior peak-SO4 (mg/L) distribution.
#
#     No full-model runs occur: this conditions the *emulator* in runstore mode.
#     """
#     sub_t_d = Path(f"dsi_subset_{name}")
#     if sub_t_d.exists():
#         shutil.rmtree(sub_t_d)
#     shutil.copytree(dsi_t_d, sub_t_d)            # the runstore template, never modified in place
#
#     dpst = pyemu.Pst(str(sub_t_d / "dsi.pst"))
#     weight_subset(dpst, species, sites=sites, t_start=t_start, t_end=t_end)
#
#     # set the conditioning targets to the synthetic truth's own values (+ noise ensemble),
#     # exactly as part1_05 did for the baseline
#     # dpst.observation_data.loc[truth.columns, 'obsval'] = truth.values[0]
#     (default IES solver -- the series deliberately does not use multimodal options; see part1_05)
#     dpst.pestpp_options['ies_num_reals'] = 1000
#     dpst.control_data.noptmax = 3
#     dpst.write(str(sub_t_d / "dsi.pst"), version=2)
#     hbd.get_bins(sub_t_d)                          # stage the binaries
#
#     m_d = Path(f"master_subset_{name}")
#     pyemu.os_utils.start_workers(
#         str(sub_t_d), "pestpp-ies", "dsi.pst",
#         num_workers=num_workers, worker_root=".", master_dir=str(m_d),
#     )
#     post = pyemu.Pst(str(m_d / "dsi.pst"))
#     oe = post.ies.obsen   # final-iteration posterior
#     return peak_so4_mgl(oe)

## The subsets we compare

Now we just call the helper for each question we want answered. Four families, each a different cut at "what is the data worth?":

**1. The headline — cations, one at a time and all together.** The baseline plus each held-back cation individually (so we can attribute worth to a *specific analyte*), and the baseline plus all cations at once (the upper bound on what the whole cation suite could buy).

In [ ]:
# TODO: run the cation data-worth cases (PENDING).
#
# results = {}
# results['baseline']  = worth_of('baseline',  COND_SPECIES, truth=truth)
# for cat in cats_present:                       # ca, na, fe, fe2
#     results[f'+{cat}'] = worth_of(f'plus_{cat}', COND_SPECIES + [cat], truth=truth)
# results['+all cations'] = worth_of('plus_allcats', COND_SPECIES + cats_present, truth=truth)

**2. Which monitoring site earns its keep?** Drop one conditioning site at a time from the baseline and watch the forecast loosen. A site whose removal barely moves the forecast is one whose sampling effort is doing little; a site whose removal opens the forecast up is load-bearing.

In [ ]:
# TODO: leave-one-site-out cases (PENDING).
#
# cond_sites = sorted(
#     obs.loc[obs.variable.isin(COND_SPECIES) & (obs.weight > 0)].obsid.unique()
# )
# for site in cond_sites:
#     keep = [s for s in cond_sites if s != site]
#     results[f'-{site}'] = worth_of(f'drop_{site}', COND_SPECIES, sites=keep, truth=truth)

**3. Which conditioning species is doing the work?** The mirror of the cation question, inside the baseline set: drop one conditioning species at a time. This is a sanity check as much as a worth measure — we *expect* dropping SO₄ or the oxidants to hurt the forecast badly, and if it does not, something is wrong with the setup.

In [ ]:
# TODO: leave-one-species-out cases (PENDING).
#
# for sp in COND_SPECIES:
#     keep = [s for s in COND_SPECIES if s != sp]
#     results[f'-{sp}'] = worth_of(f'drop_{sp}', keep, truth=truth)

**4. (Optional) Is it worth monitoring for longer?** The history period ends at the decision date (252 d). We cannot move *that* — the decision is committed then — but we can ask the related campaign question by *shortening* the window: condition on only the first half of the history period and see how much forecast skill the second half of monitoring actually added. (A genuinely *longer* window would need data past the decision date, which by construction does not exist when the decision is made — so this arm is the honest one to run.)

In [ ]:
# TODO: shorter-monitoring case (PENDING).
#
# results['short window'] = worth_of('short', COND_SPECIES, t_end=126.0, truth=truth)

## The payoff figure

One picture carries the whole notebook. For each subset we have a posterior distribution of peak SO₄ at the supply well; the worth of that subset is how its spread compares to the baseline. The figure is a **ranked data-worth chart**:

- one row per subset, sorted by forecast spread (tightest at the top);
- the bar is the posterior 5th–95th percentile range of peak SO₄ (mg/L), with the median marked;
- the **baseline** row drawn in a contrasting colour as the reference;
- the **baseline P95** as a vertical line, so the eye reads not just "narrower" but "does this subset pull the P95 — the number treatment capacity is sized to — down?".

The reading is immediate: a cation whose row is *visibly tighter* than the baseline — especially one that pulls the P95 down — is an analyte worth paying the lab for. A cation whose row is indistinguishable from the baseline is not. The leave-one-out rows say the same thing in reverse: a site or species whose removal barely widens the bar is cheap to cut.

In [ ]:
# TODO: build the ranked data-worth chart (PENDING results).
#
# import collections
# rows = []
# for name, peak in results.items():
#     rows.append(dict(
#         subset=name,
#         p05=np.percentile(peak, 5),
#         p50=np.percentile(peak, 50),
#         p95=np.percentile(peak, 95),
#         spread=np.percentile(peak, 95) - np.percentile(peak, 5),
#     ))
# dw = pd.DataFrame(rows).sort_values('spread')          # tightest forecast first
#
# fig, ax = plt.subplots(figsize=(7, 0.4 * len(dw) + 1))
# for k, (_, r) in enumerate(dw.iterrows()):
#     color = 'C1' if r.subset == 'baseline' else '0.4'
#     ax.plot([r.p05, r.p95], [k, k], color=color, lw=4, solid_capstyle='round')
#     ax.plot(r.p50, k, 'o', color='white', mec=color, ms=6, zorder=3)
# ax.axvline(dw.loc[dw.subset == 'baseline', 'p95'].iloc[0], color='r', ls='--', label='baseline P95')
# ax.set_yticks(range(len(dw)))
# ax.set_yticklabels(dw.subset)
# ax.set_xlabel('posterior peak SO$_4$ at supply well (mg/L), 5th–95th pctile')
# ax.set_title('Data worth: forecast spread by conditioning subset')
# ax.legend()
# fig.tight_layout()

A second, optional view drives the point home for the headline cation: overlay the supply-well sulfate *breakthrough ensemble* for the baseline against the best cation subset, both ribbons drawn over the supply period. If the cation is worth it, its ribbon is visibly narrower in the supply window — the same picture the optimization notebook will lean on, but here it is the *measurement* doing the tightening, not the decision.

In [ ]:
# TODO (optional): baseline vs best-cation forecast ribbons over the supply period (PENDING).
#
# fig, ax = plt.subplots(figsize=(8, 4))
# hbd.plot_forecast_ribbon(ax, oe_base,        forecast_obsnmes, color='0.5', label='baseline (5 species)')
# hbd.plot_forecast_ribbon(ax, oe_best_cation, forecast_obsnmes, color='C0',  label='baseline + best cation')
# ax.set_xlabel('time (days)'); ax.set_ylabel('SO$_4$ at supply well (mg/L)')
# ax.legend()

## Wrap-up

What this notebook bought us:

- It turned the held-back cations from a promise into an **answer**: a ranked, decision-first statement of which analytes would have tightened the supply-well sulfate forecast, and which would not.
- It did the whole thing on the emulator. The held-back cations were already simulated by the prior Monte Carlo — the data worth was a matter of re-slicing columns and re-conditioning in seconds, **with no new full-model runs**. Done conventionally this is days of ~6 min/run compute per subset; the emulation dividend is what makes it routine instead of heroic.
- The leave-one-out cases answered the budget question from both sides: not only "what should we add?" but "what could we cut?".

And the caveat that earns its keep. This is data worth *against the synthetic truth*: it tells us which measurements would help in a world the model can perfectly represent. In the field, structural error sets a ceiling on what any measurement can constrain, and a cation that looks worthless here may carry structural-error information that matters there (or vice versa). The method is exactly right; the numbers are a guide for the real campaign, not a guarantee. That gap between synthetic worth and field worth is the lesson the [real-data capstone](../part1_09_real_data_capstone/) takes up directly.